# IF25-40305: Sistem Teknologi Multimedia (STM)
## Modul 13 — Remote Photoplethysmography (rPPG): Detak Jantung dari Wajah dengan Metode POS

**Institut Teknologi Sumatera (ITERA) — Program Studi Informatika**
*Materi Kuliah & Hands-on Lab Semester Ganjil 2026/2027 · Pertemuan 13*

---

### 🎯 Capaian Pembelajaran Modul 13:
1. Menjelaskan kenapa **warna kulit berubah sedikit** setiap jantung berdetak, dan kenapa **kanal hijau** paling peka.
2. Mengubah video wajah menjadi **tiga sinyal R, G, B** (rata-rata warna di dahi dan pipi).
3. Membersihkan sinyal: **samakan jarak waktu**, **buang tren**, dan **bandpass 0,7–3 Hz** (42–180 BPM).
4. Mencari **frekuensi paling kuat** dengan **FFT** lalu mengubahnya menjadi **BPM**.
5. Memahami ide **POS** (*Plane-Orthogonal-to-Skin*) dan kenapa ia lebih **tahan gerakan & perubahan cahaya** dibanding metode GREEN.
6. Menjalankan **rPPG live** dari webcam: cari wajah sekali, lalu **ikuti dengan tracker** (tidak mendeteksi di setiap frame).

### 🧭 Cara Memakai Notebook Ini
- Di kelas, dosen menjalankan bagian bertanda **🎬 Demo Dosen** di proyektor. Anda cukup memperhatikan dan menebak.
- Di rumah, jalankan notebook dari atas ke bawah. Sel bertanda **✏️ Coba Ubah** berisi angka yang boleh Anda ganti.
- Notebook ini **tidak mengunduh apa pun**. Bagian 2–9 bisa langsung jalan memakai **sinyal contoh** buatan,
  dengan detak jantung yang sudah kita ketahui jawabannya (72 BPM).
- Untuk memakai **wajah Anda sendiri**: jalankan script **`13_rppg_pos.py`** di laptop (butuh webcam), duduk diam ±30 detik,
  tekan **`s`** untuk menyimpan sinyal ke `output/rppg_sinyal_*.csv`, lalu ubah `SUMBER = "csv"` di Bagian 3.

> 📎 Slide pendamping: *Pertemuan 13 — rPPG: Detak Jantung dari Wajah* (mctm.web.id/stm).

## 1. Persiapan

Modul ini cukup memakai **`numpy`**, **`scipy`**, dan **`matplotlib`**. Script webcam juga butuh **`opencv-contrib-python`**
(detektor wajah YuNet dari Modul 11 dan tracker KCF dari Modul 10).

```bash
pip install numpy scipy matplotlib opencv-contrib-python
```

In [ ]:
import os
import glob

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Folder kerja: bisa dari root repo (ada folder notebook_handson) atau dari dalam notebook_handson / Colab
BASE = "notebook_handson" if os.path.isdir("notebook_handson") else "."
os.makedirs(os.path.join(BASE, "output"), exist_ok=True)


def P(nama):
    return os.path.join(BASE, nama)


WARNA = {"R": "#c8102e", "G": "#2e9e6b", "B": "#1e6ec8"}
print("folder kerja:", os.path.abspath(BASE))

---
## 2. Ide Besar: Wajah Anda Berkedip Merah Setiap Jantung Berdetak

Setiap kali jantung memompa, **darah di pembuluh kecil bawah kulit bertambah sedikit**.
Darah (hemoglobin) **menyerap cahaya hijau** paling kuat, jadi kulit memantulkan hijau **sedikit lebih sedikit** saat darah datang.

Perubahannya **sangat kecil**: kurang dari 1 dari 255 tingkat kecerahan. Mata kita tidak bisa melihatnya,
tetapi kamera yang **merata-ratakan ribuan piksel** bisa.

Supaya semua orang bisa langsung mencoba, kita buat dulu **sinyal contoh**: tiga deret angka R, G, B
seolah-olah diambil dari dahi seseorang selama 30 detik, dengan **detak jantung 72 BPM** yang kita sembunyikan di dalamnya.
Isinya:

| Bahan | Besarnya | Mirip apa di dunia nyata |
|---|---|---|
| Warna kulit dasar | R 180, G 130, B 110 | kulit sawo matang di bawah lampu |
| Denyut jantung | ±0,2% (paling kuat di **hijau**) | darah datang dan pergi |
| Cahaya berubah pelan | ±3% dalam 20 detik | awan lewat, lampu meredup |
| Noise kamera | ±0,1 tingkat | bintik acak sensor |
| *(opsional)* Kepala bergerak | ±0,6% pada ±105 kali/menit | mengangguk atau berbicara |

In [ ]:
def buat_sinyal_contoh(bpm=72, detik=30, fs=30, gerak=False, seed=0):
    # Sinyal R, G, B buatan dari 'dahi' seseorang. Jawabannya kita tahu: bpm.
    rng = np.random.default_rng(seed)
    t = np.arange(0, detik, 1 / fs)
    kulit = np.array([180.0, 130.0, 110.0])                    # warna kulit rata-rata (R, G, B)
    arah_darah = np.array([0.33, 0.77, 0.53])                  # denyut paling kuat di HIJAU
    f = bpm / 60                                               # BPM -> denyut per detik (Hz)
    denyut = np.sin(2 * np.pi * f * t) + 0.3 * np.sin(2 * np.pi * 2 * f * t + 0.6)
    terang = 1 + 0.03 * np.sin(2 * np.pi * 0.05 * t)           # cahaya berubah pelan
    if gerak:
        terang = terang + 0.006 * np.sin(2 * np.pi * 1.75 * t)  # kepala bergerak ±105 kali/menit
    rgb = kulit * terang[:, None] * (1 + 0.002 * denyut[:, None] * arah_darah)
    rgb = rgb + rng.normal(0, 0.1, rgb.shape)                  # noise kamera
    waktu = t + rng.normal(0, 0.004, t.shape)                  # jam webcam tidak pernah rata sempurna
    return np.sort(waktu), rgb


waktu, rgb = buat_sinyal_contoh()
print("jumlah sampel:", len(waktu), "| durasi:", round(waktu[-1] - waktu[0], 1), "detik")
print("rata-rata R, G, B:", rgb.mean(axis=0).round(1))

### 🎬 Demo Dosen — Bisakah Anda Melihat Detaknya?

**Tebak dulu:** di grafik R, G, B mentah di bawah, bisakah Anda menghitung 72 denyut per menit (sekitar 36 denyut dalam 30 detik)?

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
for i, k in enumerate("RGB"):
    ax[i].plot(waktu, rgb[:, i], color=WARNA[k], lw=1)
    ax[i].set_ylabel(k)
ax[0].set_title("Sinyal mentah R, G, B dari dahi (sinyal contoh)")
ax[-1].set_xlabel("waktu (detik)")
plt.tight_layout(); plt.show()

**Yang terlihat:** gelombang besar dan pelan (cahaya berubah) menutupi segalanya. Denyutnya ada,
tetapi **50 kali lebih kecil** dari perubahan cahaya. Tugas kita: **mengeluarkan** denyut kecil itu.

---
## 3. Pilih Sumber Data: Sinyal Contoh atau Wajah Anda Sendiri

✏️ **Coba Ubah:**
- `SUMBER = "contoh"` memakai sinyal buatan di atas (jawabannya 72 BPM).
- `SUMBER = "csv"` memakai rekaman wajah Anda dari script `13_rppg_pos.py` (tekan `s` saat script berjalan).
- `NADI_MANUAL`: isi dengan nadi Anda sendiri. Raba nadi di pergelangan tangan, hitung **15 detik**, lalu **kalikan 4**.
  Lakukan tepat setelah merekam.

In [ ]:
SUMBER = "contoh"       # ✏️ "contoh" atau "csv"
NADI_MANUAL = None      # ✏️ misalnya 76 (hasil hitung nadi 15 detik x 4)

if SUMBER == "csv":
    berkas = sorted(glob.glob(P("output/rppg_sinyal_*.csv")), key=os.path.getmtime)
    if not berkas:
        raise FileNotFoundError("Belum ada CSV. Jalankan: python 13_rppg_pos.py, duduk diam 30 detik, tekan s.")
    tabel = np.loadtxt(berkas[-1], delimiter=",", skiprows=1)
    waktu, rgb = tabel[:, 0], tabel[:, 1:4]
    JAWABAN = NADI_MANUAL
    print("memakai", berkas[-1])
else:
    waktu, rgb = buat_sinyal_contoh(bpm=72)
    JAWABAN = 72
print(f"{len(waktu)} sampel, {waktu[-1] - waktu[0]:.1f} detik, jawaban acuan: {JAWABAN}")

---
## 4. Langkah 1 — Samakan Jarak Waktu

Webcam tidak selalu memberi frame tepat setiap 1/30 detik. Kadang 31 ms, kadang 40 ms (laptop sedang sibuk).
FFT butuh sampel yang **berjarak sama**, jadi kita **isi ulang** sinyal di titik waktu yang rata (interpolasi),
seperti menggambar ulang grafik di kertas berpetak.

In [ ]:
FS = 30.0    # sampel per detik setelah disamakan


def samakan(waktu, rgb, fs=FS):
    tt = np.arange(waktu[0], waktu[-1], 1 / fs)
    return tt, np.stack([np.interp(tt, waktu, rgb[:, k]) for k in range(3)], axis=1)


jarak_ms = np.diff(waktu) * 1000
t_rata, rgb_rata = samakan(waktu, rgb)

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].hist(jarak_ms, bins=40, color="#1e6ec8")
ax[0].set_title("Jarak antar frame asli (ms)"); ax[0].set_xlabel("ms")
ax[1].plot(waktu[:40], np.zeros(40), "o", ms=4, label="asli")
ax[1].plot(t_rata[:40], np.ones(40), "o", ms=4, label="disamakan")
ax[1].set_yticks([0, 1], ["asli", "disamakan"]); ax[1].set_title("40 sampel pertama")
plt.tight_layout(); plt.show()
print(f"jarak rata-rata {jarak_ms.mean():.1f} ms, paling pendek {jarak_ms.min():.1f}, paling panjang {jarak_ms.max():.1f}")

---
## 5. Langkah 2 — Metode GREEN: Cukup Pakai Kanal Hijau

Metode paling tua (Verkruysse dkk., 2008): ambil **kanal hijau** saja.
1. **Bagi dengan rata-ratanya** → satuannya jadi "persen perubahan", tidak peduli kulit terang atau gelap.
2. **Buang tren** (*detrend*) → garis miring karena cahaya berubah pelan dihapus.
3. Dibalik tanda minusnya: darah datang = hijau lebih gelap, kita ingin darah datang = puncak.

In [ ]:
def green(rgb):
    g = rgb[:, 1]
    return -(g / g.mean() - 1)


s_green = green(rgb_rata)
s_green_dt = signal.detrend(s_green)

fig, ax = plt.subplots(2, 1, figsize=(11, 4.5), sharex=True)
ax[0].plot(t_rata, s_green * 100, color=WARNA["G"]); ax[0].set_ylabel("% perubahan")
ax[0].set_title("Kanal hijau (dibagi rata-rata)")
ax[1].plot(t_rata, s_green_dt * 100, color=WARNA["G"]); ax[1].set_ylabel("% perubahan")
ax[1].set_title("Setelah detrend: garis miring hilang, gelombang pelan masih ada")
ax[-1].set_xlabel("waktu (detik)")
plt.tight_layout(); plt.show()

---
## 6. Langkah 3 — Bandpass 0,7–3 Hz: Penjaga Pintu

Jantung manusia (dewasa, istirahat sampai olahraga) berdetak sekitar **42–180 kali per menit**,
artinya **0,7–3 kali per detik** (Hz). Semua getaran yang lebih pelan (cahaya berubah, napas)
atau lebih cepat (noise) kita **buang** dengan *bandpass filter*, persis seperti filter audio di Modul 3.

In [ ]:
BPM_MIN, BPM_MAX = 42, 180      # ✏️ Coba Ubah: misalnya 50-120 untuk orang yang sedang duduk


def bandpass(s, fs=FS, bawah=BPM_MIN / 60, atas=BPM_MAX / 60):
    b, a = signal.butter(3, [bawah, atas], btype="band", fs=fs)
    return signal.filtfilt(b, a, s)


s_green_bp = bandpass(s_green_dt)

w, hresp = signal.freqz(*signal.butter(3, [BPM_MIN / 60, BPM_MAX / 60], btype="band", fs=FS), fs=FS, worN=2048)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1, 2]})
ax[0].plot(w * 60, np.abs(hresp), color="#1e3a5f")
ax[0].axvspan(BPM_MIN, BPM_MAX, color="#f2b705", alpha=0.2)
ax[0].set_xlim(0, 300); ax[0].set_xlabel("BPM"); ax[0].set_title("Pintu bandpass")
ax[1].plot(t_rata, s_green_bp * 100, color=WARNA["G"])
ax[1].set_xlabel("waktu (detik)"); ax[1].set_title("GREEN setelah bandpass: denyut mulai terlihat!")
plt.tight_layout(); plt.show()

---
## 7. Langkah 4 — FFT: Cari Nada Paling Kuat

Ingat Modul 5 (deteksi nada musik)? FFT memecah sinyal menjadi **campuran frekuensi**.
Detak jantung = **frekuensi yang paling kuat** di antara 0,7–3 Hz. Kalikan 60 → BPM.

> 💡 `n=4096` artinya kita "menambal nol" supaya puncaknya bisa dibaca lebih halus (bukan lompat per 6 BPM).

In [ ]:
def spektrum_bpm(s, fs=FS):
    n = 4096
    spek = np.abs(np.fft.rfft(s * np.hanning(len(s)), n=n)) ** 2
    bpm = np.fft.rfftfreq(n, 1 / fs) * 60
    pita = (bpm >= BPM_MIN) & (bpm <= BPM_MAX)
    puncak = bpm[np.argmax(spek * pita)]
    return bpm, spek, puncak


bpm_ax, spek, bpm_green = spektrum_bpm(s_green_bp)
plt.figure(figsize=(9, 3))
plt.plot(bpm_ax, spek / spek.max(), color=WARNA["G"])
plt.axvline(bpm_green, color="k", ls="--", lw=1)
plt.text(bpm_green + 3, 0.9, f"{bpm_green:.0f} BPM")
if JAWABAN:
    plt.axvline(JAWABAN, color="#c8102e", lw=1, alpha=0.6, label=f"acuan {JAWABAN} BPM"); plt.legend(frameon=False)
plt.xlim(30, 200); plt.xlabel("BPM"); plt.ylabel("kekuatan"); plt.title("Spektrum sinyal GREEN")
plt.tight_layout(); plt.show()
print(f"GREEN: {bpm_green:.1f} BPM | acuan: {JAWABAN}")

---
## 8. POS: Lihat dari Arah yang Tepat

GREEN punya kelemahan: kalau **cahaya atau gerakan** ikut naik-turun di rentang 42–180 BPM, hijau juga ikut berubah.
Lalu FFT bisa memilih **gerakan**, bukan **jantung**.

Ide POS (*Plane-Orthogonal-to-Skin*, Wang dkk., 2017) dalam bahasa sehari-hari:

1. **Bagi tiap kanal dengan rata-ratanya** (per jendela 1,6 detik). Warna kulit siapa pun menjadi (1, 1, 1).
2. Kalau **cahaya makin terang/gelap**, ketiga kanal naik-turun **bersama dan sama besar**. Denyut tidak: hijau berubah paling banyak.
3. Jadi kita **melihat dari dua arah** yang **buta terhadap perubahan bersama**:
   `S1 = G − B` dan `S2 = −2R + G + B`. Kalau R, G, B naik sama-sama, kedua arah ini **tetap nol**.
4. **Campur** kedua arah itu (dengan pengali `α` = perbandingan besar gelombangnya) supaya sisa noise saling menghapus.
5. Ulangi untuk setiap jendela 1,6 detik yang digeser, lalu **tumpuk** hasilnya.

In [ ]:
def pos(rgb, fs=FS, jendela_detik=1.6):
    n, L = len(rgb), int(jendela_detik * fs)
    h = np.zeros(n)
    for awal in range(0, n - L + 1):
        c = rgb[awal:awal + L]
        cn = c / c.mean(axis=0)                       # 1) warna kulit -> (1, 1, 1)
        s1 = cn[:, 1] - cn[:, 2]                      # 3) arah pertama:  G - B
        s2 = -2 * cn[:, 0] + cn[:, 1] + cn[:, 2]      #    arah kedua:   -2R + G + B
        p = s1 + (s1.std() / s2.std()) * s2           # 4) campur
        h[awal:awal + L] += p - p.mean()              # 5) tumpuk
    return h


# Lihat isi satu jendela 1,6 detik (detik ke-10)
L = int(1.6 * FS); awal = int(10 * FS)
c = rgb_rata[awal:awal + L]; cn = c / c.mean(axis=0)
s1 = cn[:, 1] - cn[:, 2]; s2 = -2 * cn[:, 0] + cn[:, 1] + cn[:, 2]
tj = t_rata[awal:awal + L]
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for i, k in enumerate("RGB"):
    ax[0].plot(tj, (cn[:, i] - 1) * 100, color=WARNA[k], label=k)
ax[0].set_title("1) dibagi rata-rata (%)"); ax[0].legend(frameon=False)
ax[1].plot(tj, s1 * 100, label="S1 = G-B"); ax[1].plot(tj, s2 * 100, label="S2 = -2R+G+B")
ax[1].set_title("3) dua arah pandang"); ax[1].legend(frameon=False)
ax[2].plot(tj, (s1 + s1.std() / s2.std() * s2) * 100, color="k")
ax[2].set_title("4) dicampur")
plt.tight_layout(); plt.show()

In [ ]:
s_pos = bandpass(signal.detrend(pos(rgb_rata)))
_, spek_pos, bpm_pos = spektrum_bpm(s_pos)

fig, ax = plt.subplots(2, 1, figsize=(11, 5))
ax[0].plot(t_rata, s_pos / np.abs(s_pos).max(), color="#1e3a5f")
ax[0].set_title("Sinyal POS (setelah detrend + bandpass)"); ax[0].set_xlabel("waktu (detik)")
ax[1].plot(bpm_ax, spek / spek.max(), color=WARNA["G"], label=f"GREEN: {bpm_green:.0f} BPM")
ax[1].plot(bpm_ax, spek_pos / spek_pos.max(), color="#1e3a5f", label=f"POS: {bpm_pos:.0f} BPM")
ax[1].set_xlim(30, 200); ax[1].set_xlabel("BPM"); ax[1].legend(frameon=False)
plt.tight_layout(); plt.show()
print(f"GREEN {bpm_green:.1f} | POS {bpm_pos:.1f} | acuan {JAWABAN}")

---
## 9. 🎬 Demo Dosen — Uji Gerakan: GREEN vs POS

Sekarang sinyal contoh kita beri **gerakan kepala** (mengangguk/berbicara ±105 kali per menit).
Gerakan membuat **seluruh wajah** sedikit lebih terang-gelap, jadi R, G, B berubah **bersama**.

**Tebak dulu:** metode mana yang tertipu dan menjawab ±105 BPM? GREEN, POS, atau keduanya?

✏️ **Coba Ubah:** besar gerakan ada di `buat_sinyal_contoh` (angka `0.006`). Coba 0.002 dan 0.02.

In [ ]:
def bpm_dari(rgb_mentah, waktu_mentah, metode):
    _, x = samakan(waktu_mentah, rgb_mentah)
    s = pos(x) if metode == "POS" else green(x)
    return spektrum_bpm(bandpass(signal.detrend(s)))


waktu_g, rgb_g = buat_sinyal_contoh(bpm=72, gerak=True)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2), sharey=True)
for a, metode, warna in [(ax[0], "GREEN", WARNA["G"]), (ax[1], "POS", "#1e3a5f")]:
    sumbu, sp, hasil = bpm_dari(rgb_g, waktu_g, metode)
    a.plot(sumbu, sp / sp.max(), color=warna)
    a.axvline(72, color="#c8102e", lw=1, alpha=0.6); a.axvline(105, color="gray", ls=":", lw=1)
    a.set_title(f"{metode}: {hasil:.0f} BPM (jantung 72, gerakan 105)"); a.set_xlim(30, 200); a.set_xlabel("BPM")
plt.tight_layout(); plt.show()

**Kenapa begitu?** Gerakan ini mengubah R, G, B **bersama dan sama besar** (setelah dibagi rata-rata).
GREEN hanya melihat hijau, jadi ia ikut naik-turun. POS melihat dari arah `G − B` dan `−2R + G + B`,
yang **nol** untuk perubahan bersama. Gerakannya hilang, jantungnya tetap terlihat.

> ⚠️ Di dunia nyata gerakan tidak selalu "bersama dan sama besar" (misalnya kilau keringat, bayangan hidung yang berpindah).
> Karena itu POS **lebih tahan**, bukan **kebal**.

---
## 10. BPM dari Waktu ke Waktu (Jendela Geser)

Aplikasi live tidak menunggu 30 detik. Ia menghitung BPM dari **10 detik terakhir**, lalu menggeser jendelanya
setiap 1 detik. Hasilnya **grafik BPM** seperti di smartwatch.

✏️ **Coba Ubah:** `JENDELA` (detik). Jendela pendek = cepat bereaksi tapi lebih goyang.

In [ ]:
JENDELA = 10      # ✏️ detik
GESER = 1         # detik

t_rata, rgb_rata = samakan(waktu, rgb)
hasil = {"GREEN": [], "POS": []}
waktu_tengah = []
for mulai in np.arange(0, t_rata[-1] - t_rata[0] - JENDELA, GESER):
    sel = (t_rata - t_rata[0] >= mulai) & (t_rata - t_rata[0] < mulai + JENDELA)
    waktu_tengah.append(mulai + JENDELA)
    for metode in hasil:
        s = pos(rgb_rata[sel]) if metode == "POS" else green(rgb_rata[sel])
        hasil[metode].append(spektrum_bpm(bandpass(signal.detrend(s)))[2])

plt.figure(figsize=(10, 3.2))
plt.plot(waktu_tengah, hasil["GREEN"], "o-", color=WARNA["G"], label="GREEN")
plt.plot(waktu_tengah, hasil["POS"], "s-", color="#1e3a5f", label="POS")
if JAWABAN:
    plt.axhline(JAWABAN, color="#c8102e", lw=1, label=f"acuan {JAWABAN}")
plt.xlabel("detik ke-"); plt.ylabel("BPM"); plt.legend(frameon=False)
plt.title(f"BPM per jendela {JENDELA} detik")
plt.tight_layout(); plt.show()
for metode, nilai in hasil.items():
    print(f"{metode:5s}: median {np.median(nilai):.1f} BPM, rentang {min(nilai):.0f}-{max(nilai):.0f}")

---
## 11. Live dari Webcam: `13_rppg_pos.py`

Script ini menjalankan **semua langkah di atas** setiap 0,5 detik, ditambah satu trik supaya FPS tetap tinggi:

| Tahap | Alat | Kapan dijalankan |
|---|---|---|
| Cari wajah | **YuNet** (Modul 11), memberi kotak + 5 titik (mata, hidung, mulut) | **Hanya** di awal, saat tracker kehilangan wajah, atau saat kotak bergeser jauh |
| Ikuti wajah | **Tracker KCF** (Modul 10) | Setiap frame (jauh lebih ringan) |
| ROI | 3 kotak: dahi + 2 pipi, posisinya dihitung dari jarak antar mata | Ikut bergeser bersama kotak wajah |
| BPM | samakan waktu → POS/GREEN → detrend → bandpass → FFT | Setiap 0,5 detik, dari 10 detik terakhir |

```bash
python 13_rppg_pos.py                     # webcam
python 13_rppg_pos.py video/wajah.mp4     # atau file video wajah Anda (CSV disimpan otomatis)
```

**Tombol:** `q` keluar · `m` ganti POS/GREEN · `d` paksa deteksi ulang · `r` mulai ulang · `s` simpan CSV · `k` kotak ROI

✏️ **Coba Ubah di script:**
- `BATAS_GESER = 0.35` → coba 0.1 (sering deteksi ulang) dan 1.0 (hampir tidak pernah). Lihat angka "deteksi wajah: …x".
- `JENDELA = 10.0` → coba 5 dan 20.
- Duduk diam 20 detik, lalu bicara atau mengangguk: bandingkan POS dan GREEN dengan tombol `m`.

> ⚠️ Di macOS, izinkan Terminal/VS Code memakai kamera: *System Settings ➜ Privacy & Security ➜ Camera*.
> 🩺 **Bukan alat medis.** Angka ini untuk belajar, bukan untuk diagnosis.

---
## 12. Rangkuman

1. Setiap detak jantung, darah di bawah kulit bertambah sedikit → kulit memantulkan **hijau sedikit lebih sedikit** (< 1 tingkat dari 255).
2. Video wajah → **rata-rata R, G, B** di dahi & pipi → tiga sinyal.
3. Bersihkan: **samakan jarak waktu** → **detrend** → **bandpass 0,7–3 Hz** (42–180 BPM).
4. **FFT** → frekuensi terkuat × 60 = **BPM**.
5. **GREEN** sederhana tetapi mudah tertipu cahaya & gerakan. **POS** melihat dari dua arah yang buta terhadap perubahan bersama R, G, B.
6. Live: **deteksi sekali, lalu lacak**. Deteksi ulang hanya kalau tracker hilang atau wajah bergeser jauh.

### 🧭 Lembar Contekan
| Tujuan | Kode |
|---|---|
| Samakan waktu | `np.interp(tt, waktu, x)` dengan `tt = np.arange(t0, t1, 1/30)` |
| Buang tren | `signal.detrend(s)` |
| Bandpass | `signal.filtfilt(*signal.butter(3, [0.7, 3.0], btype="band", fs=30), s)` |
| BPM | `rfftfreq(n, 1/fs)[argmax(abs(rfft(s, n))**2)] * 60` (di dalam 0,7–3 Hz) |
| POS (inti) | `cn = c / c.mean(0); s1 = G-B; s2 = -2R+G+B; h = s1 + std(s1)/std(s2) * s2` |
| Deteksi wajah | `cv2.FaceDetectorYN.create("model/face_detection_yunet_2023mar.onnx", "", (w, h))` |
| Lacak wajah | `tr = cv2.TrackerKCF_create(); tr.init(frame, kotak); ok, kotak = tr.update(frame)` |

> 🔜 **Pertemuan berikutnya:** Multimedia Networking & Streaming — bagaimana video sampai ke HP Anda tanpa macet.